# agentic-civic-resolution-app 
Notebook 4: Multi-Agent Orchestrator

**Chains all three agents into a single workflow:**

```
Raw Complaint Text
       │
       ▼
 ┌─────────────────────┐
 │ Agent 1: Classifier  │  → category, subcategory, confidence
 └─────────────────────┘
       │
       ▼
 ┌─────────────────────┐
 │ Agent 2: Severity    │  → severity_score, priority_label, sla_hours
 └─────────────────────┘
       │
       ▼
 ┌─────────────────────┐
 │ Agent 3: Router      │  → dept_code, officer_tier, action_plan, escalate
 └─────────────────────┘
       │
       ▼
  Final CivicOps Ticket (written to Delta)
```

 **Deliverables:**
 - `CivicOpsTicket` — unified output schema
 - `CivicOpsOrchestrator` — single-call pipeline
 - Batch processing from Silver Delta table
 - Results written to `civicops.gold.processed_tickets`

**COMMAND:**
 
%pip install databricks-langchain mlflow pydantic --quiet

dbutils.library.restartPython()

## 0. Run Agent Notebooks (loads all three agents into scope)

 In Databricks, `%run` imports all names from the target notebook.

In [0]:
%run ./agent_01_classifier


In [0]:
%run ./agent_02_severity


In [0]:
%run ./agent_03_router

## 1. Unified Ticket Schema

In [0]:
import uuid
from datetime import datetime, timedelta
from pydantic import BaseModel, Field
from typing import Optional

class CivicOpsTicket(BaseModel):
    # Identifiers
    ticket_id:          str = Field(default_factory=lambda: f"TKT-{uuid.uuid4().hex[:8].upper()}")
    created_at:         datetime = Field(default_factory=datetime.utcnow)

    # Input
    raw_complaint:      str

    # Agent 1 — Classification
    category:           str
    subcategory:        str
    classification_confidence: float

    # Agent 2 — Severity
    severity_score:     int
    priority_label:     str
    sla_hours:          int
    sla_deadline:       datetime
    health_risk:        bool
    infrastructure_risk: bool
    affected_estimate:  str

    # Agent 3 — Routing
    dept_code:          str
    dept_name:          str
    dept_contact:       str
    officer_tier:       str
    escalate:           bool
    escalation_dept:    Optional[str]
    action_plan:        list[str]
    field_visit_needed: bool
    notify_citizen:     bool

    # Pipeline metadata
    pipeline_version:   str = "1.0.0"
    processing_ms:      Optional[int] = None

    def summary(self) -> str:
        lines = [
            f"╔══ CivicOps Ticket: {self.ticket_id} ══",
            f"║ Complaint  : {self.raw_complaint}",
            f"║ Category   : {self.category} > {self.subcategory} (conf: {self.classification_confidence:.2f})",
            f"║ Severity   : {self.severity_score}/5 — {self.priority_label}",
            f"║ SLA        : Resolve by {self.sla_deadline.strftime('%Y-%m-%d %H:%M')} UTC ({self.sla_hours}h)",
            f"║ Department : [{self.dept_code}] {self.dept_name}",
            f"║ Officer    : {self.officer_tier}",
            f"║ Escalate   : {'YES → ' + (self.escalation_dept or '') if self.escalate else 'No'}",
            f"║ Field Visit: {'Required' if self.field_visit_needed else 'Not required'}",
            f"║ Action Plan:",
        ]
        for i, step in enumerate(self.action_plan, 1):
            lines.append(f"║   {i}. {step}")
        lines.append(f"╚{'═' * 50}")
        return "\n".join(lines)

## 2. Orchestrator Class

In [0]:
import json, re, requests
import time, requests, mlflow
from pydantic import BaseModel, Field
from databricks_langchain import ChatDatabricks

def discover_llm_endpoint() -> str:
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}"}
    resp    = requests.get(f"{host}/api/2.0/serving-endpoints", headers=headers)
    ready   = [e for e in resp.json().get("endpoints", []) if e.get("state", {}).get("ready") == "READY"]
    for kw in ["llama", "dbrx", "mixtral", "mpt"]:
        for e in ready:
            if kw in e["name"].lower():
                return e["name"]
    return ready[0]["name"]

LLM_ENDPOINT = discover_llm_endpoint()
print(f"✓ Endpoint: {LLM_ENDPOINT}")


In [0]:
def ensure_mlflow_experiment(experiment_path: str) -> str:
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    parts = experiment_path.strip("/").split("/")
    for depth in range(1, len(parts)):
        folder = "/" + "/".join(parts[:depth])
        resp   = requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=headers, json={"path": folder})
        body   = resp.json()
        if resp.status_code == 200:
            print(f"  ✓ folder: {folder}")
        elif body.get("error_code") == "RESOURCE_ALREADY_EXISTS":
            print(f"  ✓ exists: {folder}")
        else:
            raise RuntimeError(f"mkdirs failed for {folder}: {body}")

    exp = mlflow.set_experiment(experiment_path)
    return exp.experiment_id

EXPERIMENT_ID = ensure_mlflow_experiment("/civicops/orchestrator")
print(f"✓ MLflow experiment ready (id={EXPERIMENT_ID})")



In [0]:
class CivicOpsOrchestrator:
    """
    Sequential multi-agent pipeline.
    Each agent's output feeds into the next agent's input.
    """

    def __init__(self):
        self.classifier = classifier_agent   # from agent_01_classifier
        self.scorer     = severity_agent     # from agent_02_severity
        self.router     = router_agent       # from agent_03_router

    # ── Single complaint ──────────────────────────────────────────────────
    def process(self, complaint_text: str) -> CivicOpsTicket:
        """
        Full pipeline: raw text → CivicOpsTicket.

        Args:
            complaint_text: Raw complaint from a citizen.

        Returns:
            CivicOpsTicket with all fields populated.
        """
        start = time.time()

        with mlflow.start_run(run_name="orchestrate"):
            mlflow.log_param("complaint", complaint_text[:200])

            # ── Step 1: Classify ──────────────────────────────────────────
            cls = self.classifier.classify(complaint_text)
            mlflow.log_params({
                "category":    cls.category,
                "subcategory": cls.subcategory,
                "cls_conf":    cls.confidence,
            })

            # ── Step 2: Score Severity ────────────────────────────────────
            sev = self.scorer.score(
                complaint_text,
                category=cls.category,
                subcategory=cls.subcategory,
            )
            mlflow.log_params({
                "severity_score": sev.severity_score,
                "priority_label": sev.priority_label,
                "health_risk":    sev.health_risk,
            })

            # ── Step 3: Route ─────────────────────────────────────────────
            route = self.router.route(
                complaint_text=complaint_text,
                category=cls.category,
                subcategory=cls.subcategory,
                severity_score=sev.severity_score,
                health_risk=sev.health_risk,
                infrastructure_risk=sev.infrastructure_risk,
            )

            elapsed_ms = int((time.time() - start) * 1000)
            mlflow.log_metric("processing_ms", elapsed_ms)

            # ── Assemble Ticket ───────────────────────────────────────────
            ticket = CivicOpsTicket(
                raw_complaint=complaint_text,
                # Classification
                category=cls.category,
                subcategory=cls.subcategory,
                classification_confidence=cls.confidence,
                # Severity
                severity_score=sev.severity_score,
                priority_label=sev.priority_label,
                sla_hours=sev.sla_hours,
                sla_deadline=datetime.utcnow() + timedelta(hours=sev.sla_hours),
                health_risk=sev.health_risk,
                infrastructure_risk=sev.infrastructure_risk,
                affected_estimate=sev.affected_estimate,
                # Routing
                dept_code=route.dept_code,
                dept_name=route.dept_name,
                dept_contact=route.dept_contact,
                officer_tier=route.officer_tier,
                escalate=route.escalate,
                escalation_dept=route.escalation_dept,
                action_plan=route.action_plan,
                field_visit_needed=route.field_visit_needed,
                notify_citizen=route.notify_citizen,
                processing_ms=elapsed_ms,
            )

            mlflow.log_param("ticket_id", ticket.ticket_id)

        return ticket

    # ── Batch from Delta ──────────────────────────────────────────────────
    def process_batch(self, complaints: list[str]) -> list[CivicOpsTicket]:
        """Process a list of complaints sequentially. Add ThreadPoolExecutor for parallelism."""
        return [self.process(c) for c in complaints]

    def process_delta_batch(
        self,
        source_table: str = "civicops.silver.complaints",
        limit: int = 100,
        unprocessed_only: bool = True,
    ) -> list[CivicOpsTicket]:
        """
        Pull unprocessed complaints from Silver Delta → run pipeline → return tickets.

        Args:
            source_table:      Silver Delta table name.
            limit:             Max complaints to process in one run.
            unprocessed_only:  Filter to rows not yet in gold.processed_tickets.
        """
        from pyspark.sql import functions as F

        silver_df = spark.table(source_table)

        if unprocessed_only:
            try:
                processed_ids = (
                    spark.table("civicops.gold.processed_tickets")
                    .select("raw_complaint")
                )
                silver_df = silver_df.join(processed_ids, 
                    silver_df.complaint_text == processed_ids.raw_complaint, 
                    how="left_anti"
                )
                
            except Exception:
                pass  # Gold table doesn't exist yet on first run

        # Pull complaint text — use descriptor + complaint_type as the text
        rows = (
            silver_df
            .filter(F.col("complaint_type").isNotNull())
            .withColumn("complaint_text",
                F.concat_ws(" near ", F.col("complaint_type"), F.col("street_name")))
            .select("complaint_id", "complaint_text")
            .limit(limit)
            .collect()
        )

        tickets = []
        for row in rows:
            try:
                ticket = self.process(row["complaint_text"])
                tickets.append(ticket)
            except Exception as e:
                print(f"  ⚠ Skipped {row['complaint_id']}: {e}")

        return tickets


orchestrator = CivicOpsOrchestrator()
print("✓ CivicOps Orchestrator ready.")

## 3. Live Demo — Example Input

In [0]:
DEMO_COMPLAINTS = [
    "Garbage overflow near Whitefield",
    "Exposed live electrical wires near the school playground on Park Street",
    "Water pipeline burst on Main Street — entire road flooded since 2 days",
    "Loud music from the bar next door every night after 11 PM",
    "Large pothole on the main road causing accidents near City Hospital",
]

for complaint in DEMO_COMPLAINTS:
    ticket = orchestrator.process(complaint)
    print(ticket.summary())
    print()

## 4. Write Results to Gold Delta Table

In [0]:
import pandas as pd
from pyspark.sql import functions as F

def tickets_to_spark(tickets: list[CivicOpsTicket]):
    """Convert ticket list → Spark DataFrame for Delta write."""
    rows = []
    for t in tickets:
        d = t.model_dump()
        d["action_plan"]      = json.dumps(d["action_plan"])       # store as JSON string
        d["created_at"]       = d["created_at"].isoformat()
        d["sla_deadline"]     = d["sla_deadline"].isoformat()
        d["escalation_dept"]  = d.get("escalation_dept") or ""
        rows.append(d)
    return spark.createDataFrame(pd.DataFrame(rows))


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.gold")

In [0]:

# Process demo complaints and write to Gold
demo_tickets = [orchestrator.process(c) for c in DEMO_COMPLAINTS]
tickets_sdf  = tickets_to_spark(demo_tickets)

(
    tickets_sdf.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("civicops.gold.processed_tickets")
)

print(f"✓ {len(demo_tickets)} tickets written to civicops.gold.processed_tickets")
display(spark.table("civicops.gold.processed_tickets"))

## 5. Batch Process Silver Table (unprocessed complaints)

In [0]:
# Uncomment to run batch processing on real Silver data

batch_tickets = orchestrator.process_delta_batch(
     source_table="civicops.silver.complaints",
     limit=50,
     unprocessed_only=True,
 )
if batch_tickets:
     batch_sdf = tickets_to_spark(batch_tickets)
     batch_sdf.write.format("delta").mode("append") \
         .option("mergeSchema", "true") \
         .saveAsTable("civicops.gold.processed_tickets")
     print(f"✓ {len(batch_tickets)} batch tickets written.")

## 6. Gold Table Stats

In [0]:
gold = spark.table("civicops.gold.processed_tickets")

print(f"Total tickets: {gold.count()}")

In [0]:
display(
    gold.groupBy("priority_label")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("sla_hours"), 1).alias("avg_sla_hours"),
        F.sum(F.col("escalate").cast("int")).alias("escalations"),
    )
    .orderBy(F.desc("count"))
)
